# MiMo 工具调用返回格式测试

In [8]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import pprint

load_dotenv()

MODEL = "mimo-v2.5-pro"
MIMO_API_KEY = os.environ.get("MIMO_API_KEY")

client = OpenAI(
    api_key=MIMO_API_KEY,
    base_url="https://api.xiaomimimo.com/v1",
)

print(f"✓ 模型: {MODEL}")
print(f"✓ API Key: {MIMO_API_KEY[:20] if MIMO_API_KEY else 'Not set'}...")

✓ 模型: mimo-v2.5-pro
✓ API Key: sk-sv5nj86hqn3akzako...


## 工具定义

In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Execute a bash command and return its output.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
            },
        },
    }
]

print("工具定义:")
pprint.pprint(TOOLS)

工具定义:
[{'function': {'description': 'Execute a bash command and return its output.',
               'name': 'run_command',
               'parameters': {'properties': {'command': {'type': 'string'}},
                              'required': ['command'],
                              'type': 'object'}},
  'type': 'function'}]


## 工具调用测试 - 完整上下文

In [11]:
test_tasks = [
    "列出当前目录的所有文件",
    "显示当前时间",
    "查看磁盘使用情况",
]

for i, task in enumerate(test_tasks, 1):
    print(f"\n{'='*70}")
    print(f"测试 {i}: {task}")
    print(f"{'='*70}")

    request_payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are MiMo, an AI assistant developed by Xiaomi."},
            {"role": "user", "content": task},
        ],
        "tools": TOOLS,
    }

    print("\n▶ 发送请求体:")
    print(json.dumps(request_payload, indent=2, ensure_ascii=False))

    try:
        response = client.chat.completions.create(**request_payload)

        print("\n◀ 完整响应体:")
        print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

        msg = response.choices[0].message
        print(f"\n── 解析摘要 ──")
        print(f"role: {msg.role}")
        print(f"content: {msg.content}")
        print(f"tool_calls 数量: {len(msg.tool_calls) if msg.tool_calls else 0}")

        if msg.tool_calls:
            for j, tc in enumerate(msg.tool_calls, 1):
                print(f"\n  工具调用 #{j}")
                print(f"  id:   {tc.id}")
                print(f"  name: {tc.function.name}")
                print(f"  arguments (raw type={type(tc.function.arguments).__name__}): {tc.function.arguments}")
                try:
                    args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
                    print(f"  arguments (parsed): {args}")
                    print(f"  → command: {args.get('command', 'N/A')}")
                except Exception as e:
                    print(f"  ✗ 参数解析失败: {type(e).__name__}: {e}")
        else:
            print(f"\n  [无工具调用] 模型直接回答: {(msg.content or '')[:120]}")

    except Exception as e:
        print(f"\n✗ 请求失败: {type(e).__name__}: {str(e)[:300]}")


测试 1: 列出当前目录的所有文件

▶ 发送请求体:
{
  "model": "mimo-v2.5-pro",
  "messages": [
    {
      "role": "system",
      "content": "You are MiMo, an AI assistant developed by Xiaomi."
    },
    {
      "role": "user",
      "content": "列出当前目录的所有文件"
    }
  ],
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "run_command",
        "description": "Execute a bash command and return its output.",
        "parameters": {
          "type": "object",
          "properties": {
            "command": {
              "type": "string"
            }
          },
          "required": [
            "command"
          ]
        }
      }
    }
  ]
}

◀ 完整响应体:
{
  "id": "9f4b69bfa66146f5b4af02cbae2d85d8",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": nu

In [12]:
# 最后一次响应的完整 JSON 转储
print("[最后一次响应的完整数据]")
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False)[:2000])
print("..." if len(json.dumps(response.model_dump())) > 2000 else "")

[最后一次响应的完整数据]
{
  "id": "25213cd429dd44dcb127b25d6cc819cb",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_9deb22eba47843c2b0d03766",
            "function": {
              "arguments": "{\"command\": \"df -h\"}",
              "name": "run_command"
            },
            "type": "function"
          }
        ],
        "reasoning_content": "用户想查看磁盘使用情况。我需要运行一个命令来获取磁盘使用信息。在Linux中，常用的命令有`df`（显示文件系统的磁盘空间使用情况）和`du`（估计文件空间的使用情况）。用户没有指定具体需求，所以我先运行一个基本的`df`命令，比如`df -h`，这样输出会更易读。如果用户还需要更详细的信息，比如特定目录的使用情况，我可以再运行`du`命令。先从`df -h`开始吧。"
      }
    }
  ],
  "created": 1781348892,
  "model": "mimo-v2.5-pro",
  "object": "chat.completion",
  "moderation": null,
  "service_tier": null,
  "system_fingerprint"